In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error

# Load the dataset
df = pd.read_csv("car_fuel_efficiency_2026.csv")

# Fill missing values with zero
df = df.fillna(0)

# Split exactly as instructed
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

# Separate target from features
target = "fuel_efficiency_mpg"

y_train = df_train[target].values
y_val = df_val[target].values
y_test = df_test[target].values

# Convert feature dataframes to dictionaries
train_dicts = df_train.drop(
    columns=[target]
).to_dict(orient="records")

val_dicts = df_val.drop(
    columns=[target]
).to_dict(orient="records")

test_dicts = df_test.drop(
    columns=[target]
).to_dict(orient="records")

# Apply DictVectorizer
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)
X_test = dv.transform(test_dicts)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (6000, 16)
Validation: (2000, 16)
Testing: (2000, 16)


QUESTION 1

In [3]:
from sklearn.tree import DecisionTreeRegressor, export_text

# Train decision tree with maximum depth of 1
dt = DecisionTreeRegressor(max_depth=1, random_state=1)
dt.fit(X_train, y_train)

# Identify the feature used for the first split
feature_index = dt.tree_.feature[0]
feature_name = dv.get_feature_names_out()[feature_index]

print("Splitting feature:", feature_name)

# Display the tree's decision rule
print(export_text(dt, feature_names=list(dv.get_feature_names_out())))

Splitting feature: model_year
|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



QUESTION 2

In [4]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

# Train random forest model
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Predict on validation dataset
y_pred = rf.predict(X_val)

# Calculate RMSE
rmse = np.sqrt(mean_squared_error(y_val, y_pred))

print("Validation RMSE:", round(rmse, 3))

Validation RMSE: 1.833


In [14]:
print("Training shape:", X_train.shape)
print("Validation shape:", X_val.shape)

print("Training target mean:", y_train.mean())
print("Validation target mean:", y_val.mean())

print("Feature count:", len(dv.get_feature_names_out()))

Training shape: (6000, 16)
Validation shape: (2000, 16)
Training target mean: 30.01151666666667
Validation target mean: 29.98825
Feature count: 16


QUESTION 3

In [5]:
n_estimators_values = [10, 50, 100, 150]

for n in n_estimators_values:

    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)

    y_pred = rf.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred))

    print(f"n_estimators={n}, RMSE={rmse:.3f}")

n_estimators=10, RMSE=1.833
n_estimators=50, RMSE=1.774
n_estimators=100, RMSE=1.770
n_estimators=150, RMSE=1.769


QUESSTION 4

In [6]:
depth_values = [10, 15, 20, 25]
n_estimators_values = [10, 50, 100, 150]

depth_results = {}

for depth in depth_values:
    rmse_scores = []

    for n in n_estimators_values:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        y_pred = rf.predict(X_val)

        score = np.sqrt(
            mean_squared_error(y_val, y_pred)
        )

        rmse_scores.append(score)

    mean_rmse = np.mean(rmse_scores)
    depth_results[depth] = mean_rmse

    print(
        f"max_depth={depth}, "
        f"Mean RMSE={mean_rmse:.3f}"
    )

best_depth = min(depth_results, key=depth_results.get)

print("\nBest max_depth:", best_depth)

max_depth=10, Mean RMSE=1.754
max_depth=15, Mean RMSE=1.784
max_depth=20, Mean RMSE=1.787
max_depth=25, Mean RMSE=1.786

Best max_depth: 10


QUESTION 5

In [7]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# Extract feature importance
feature_importance = pd.Series(
    rf.feature_importances_,
    index=dv.get_feature_names_out()
)

# Compare the four options
features = [
    "vehicle_weight",
    "horsepower",
    "acceleration",
    "engine_displacement"
]

print(feature_importance[features].sort_values(ascending=False))

vehicle_weight         0.203313
horsepower             0.078271
engine_displacement    0.061281
acceleration           0.054998
dtype: float64


QUESTION 6

In [11]:
%pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
  Using cached xgboost-3.4.1-py3-none-win_amd64.whl.metadata (2.0 kB)
Using cached xgboost-3.4.1-py3-none-win_amd64.whl (48.9 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [13]:
import xgboost as xgb
from sklearn.metrics import mean_squared_error

# Create DMatrix objects
features = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=features
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=features
)

# Create watchlist
watchlist = [
    (dtrain, "train"),
    (dval, "val")
]

# Model parameters
xgb_params = {
    "eta": 0.3,
    "max_depth": 6,
    "min_child_weight": 1,
    "objective": "reg:squarederror",
    "nthread": 8,
    "seed": 1,
    "verbosity": 1
}

# Compare learning rates
for eta in [0.3, 0.1]:
    xgb_params["eta"] = eta

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )

    predictions = model.predict(dval)

    rmse = np.sqrt(
        mean_squared_error(y_val, predictions)
    )

    print(f"eta={eta}, Validation RMSE={rmse:.3f}")

eta=0.3, Validation RMSE=1.826
eta=0.1, Validation RMSE=1.725
